<a href="https://colab.research.google.com/github/Moulicodes/movie-recommendation-system/blob/main/movie_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df_movies = pd.read_csv('/content/tmdb_5000_movies.csv')

In [ ]:
df_credits = pd.read_csv('/content/tmdb_5000_credits.csv')

In [ ]:
df_movies.head(1)

In [ ]:
df_credits.head(1)

In [ ]:
df = df_movies.merge(df_credits, on = 'title')
df.columns

In [ ]:
df = df[['title', 'genres', 'keywords', 'overview', 'tagline', 'cast', 'crew', 'id']]

In [ ]:
df_movies.head(1)['genres'].values[0]

In [ ]:
df.isnull().sum()

In [ ]:
df['overview'] = df['overview'].fillna('')
df['tagline'] = df['tagline'].fillna('')

In [ ]:
def transform_listofdicts(listofdicts):
  l = []
  for dict in listofdicts:
    l.append(dict['name'].replace(" ", ""))
  return l

In [ ]:
import ast

df['keywords'] = df['keywords'].apply(ast.literal_eval).apply(transform_listofdicts)
df['genres'] = df['genres'].apply(ast.literal_eval).apply(transform_listofdicts)

In [ ]:
df['overview'] = df['overview'].apply(lambda x: x.split())
df['tagline'] = df['tagline'].apply(lambda x: x.split())
df[['overview', 'tagline']].head()

In [ ]:
df.head(1)

In [ ]:
def get_crew(listofdicts):
  l = []
  for dict in listofdicts:
    if dict['job'] == 'Director':
      l.append(dict['name'].replace(" ", ""))
  return l

In [ ]:
df['cast'] = df['cast'].apply(ast.literal_eval).apply(transform_listofdicts).apply(lambda x: x[0:4])
df['crew'] = df['crew'].apply(ast.literal_eval).apply(get_crew)

In [ ]:
df.shape

In [ ]:
df['tags'] = df['genres'] + df['keywords'] + df['overview'] + df['tagline'] + df['cast'] + df['crew']
df['tags'] = df['tags'].apply(lambda x: " ".join(x))

In [ ]:
df.head()

1. TF-IDF Vectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words = 'english')
tfidf_matrix = tfidf.fit_transform(df['tags'])
tfidf_matrix.shape

Because we are using tfidf vectorizer there's no need to explicitly remove stop words, punctuations or convert texts to lower case as it will handle all these functions by itself. But we still need to explicitly implement space collapsing wherever necessary. For example, removing the space between first and last name.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

tfidf_cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
tfidf_cosine_sim_rounded = tfidf_cosine_sim.round(3)
tfidf_cosine_sim_rounded.shape

In [ ]:
np.where((tfidf_cosine_sim_rounded[2] > 0.1) & (tfidf_cosine_sim_rounded[2] < 0.99))[0]

In [ ]:
tfidf_cosine_sim_rounded[2][(tfidf_cosine_sim_rounded[2] > 0.1) & (tfidf_cosine_sim_rounded[2] < 0.99)]

In [ ]:
df.iloc[[2]]

In [ ]:
df.iloc[[  11,   29,  147,  277, 1344, 1745, 2360, 3165, 3288, 4076, 4345]]['title']

**Problem with tfidf vectorizer:**

It penalizes the words present in genres, keywords, cast and crew if they occur very frequently in multiple movies. But the more matching the genres, keywords, cast and crew of various movies, the greater should be the similarity scores between them. But because tfidf penalizes the frequently occuring words, it behaves counter-productive to our goal.**bold text**

2. Count Vectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(stop_words = 'english')
count_matrix = count_vectorizer.fit_transform(df['tags'])
count_matrix.shape

In [ ]:
count_cosine_sim = cosine_similarity(count_matrix, count_matrix)
count_cosine_sim_rounded = count_cosine_sim.round(3)
count_cosine_sim_rounded.shape

In [ ]:
np.where((count_cosine_sim_rounded[2] > 0.1) & (count_cosine_sim_rounded[2] < 0.99))[0]

In [ ]:
count_cosine_sim_rounded[2][(count_cosine_sim_rounded[2] > 0.1) & (count_cosine_sim_rounded[2] < 0.99)]

In [ ]:
df.iloc[[2]]

In [ ]:
df.iloc[np.where((count_cosine_sim_rounded[2] > 0.1) & (count_cosine_sim_rounded[2] < 0.99))[0]]['title']

**Problem with count vectorizer:**

Consider two completely unrelated action movies.

Movie A (James Bond): "A man goes on a mission to save the world from danger."

Movie B (Avatar): "A man on a foreign planet tries to save his new world from danger."

Both movies heavily feature highly common narrative words: "man", "save", "world", "danger"

So, both of these movies are likely to have high similarity scores even though they are completely unrelated films

A better solution:
*   Use count vectorizer for genres, keywords, cast, crew and columns where the greater the occurence of similar words in various movies higher is the similarity between them.
*   Use tf-idf vectorizer for overview, tagline and columns where the context of the entire text and the excluisivity of certain words decide the similarity scores.
*   Before applying tf-idf vectorizer, apply stemming to the columns we have considered for tf_idf vectorization.




In [ ]:
df['cv_col'] = df['genres'] + df['keywords'] + df['cast'] + df['crew']
df['tfidf_col'] = df['overview'] + df['tagline']

In [ ]:
df['cv_col'] = df['cv_col'].apply(lambda x: " ".join(x))
df['tfidf_col'] = df['tfidf_col'].apply(lambda x: " ".join(x))

In [ ]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()
df['tfidf_col'] = df['tfidf_col'].apply(lambda x: " ".join([ps.stem(word) for word in x.split()]))

In [ ]:
df['tfidf_col'].head(1).values

In [ ]:
count_vec = CountVectorizer(stop_words = 'english')
count_mat = count_vec.fit_transform(df['cv_col'])

tfidf_vec = TfidfVectorizer(stop_words = 'english')
tfidf_mat = tfidf_vec.fit_transform(df['tfidf_col'])

In [ ]:
from scipy.sparse import hstack

combined_mat = hstack([count_mat, tfidf_mat])
hybrid_sim = cosine_similarity(combined_mat, combined_mat).round(3)

In [ ]:
def recommend_movies(movie, top_n=5):
  movie_index = df[df['title'] == movie].index[0]

  similarity_scores = hybrid_sim[movie_index]

  # [::-1] reverses the array so the highest scores come first
  # We dont start from 0 (i.e, [1:top_n+1]) because the highest score (1.0) is the movie itself!
  sorted_indices = np.argsort(similarity_scores)[::-1][1:top_n+1]
  print(f"Top {top_n} Recommended Movies:")
  return df.iloc[sorted_indices][['title']]

In [ ]:
recommend_movies('Avatar')

In [ ]:
recommend_movies("Pirates of the Caribbean: At World's End")

In [ ]:
recommend_movies('Transformers: Age of Extinction')

In [ ]:
import pickle

pickle.dump(df, open("movies_df.pkl", "wb"))
pickle.dump(hybrid_sim, open("hybrid_sim.pkl", "wb"))